# Design counts

**Purpose.** Computes, from the stored files, the counts that
[`docs/research_questions.md`](../../docs/research_questions.md) cites
for the design of RQ1, RQ2 and RQ3. It only counts: no model is
trained and nothing is written. Run it again whenever the presences,
the expert polygons, the stored embeddings or a background pool change.

Inputs: the cleaned presences (`inat_dealbata_clean.csv`), the raw
iNaturalist download, the cleaned expert polygons, the RQ1 pilot
embeddings and the two background pools.

In [ ]:
import sys
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.background import load_background
from src.embeddings import load_pilot_embeddings
from src.grid import pixel_id

clean = pd.read_csv(ROOT / "data/interim/citizen_science/inat_dealbata_clean.csv")
expert = gpd.read_file(ROOT / "data/interim/expert_reference/expert_polygons_clean.gpkg").to_crs(32629)
pilot, pilot_emb = load_pilot_embeddings(ROOT / "data/processed/embeddings/tessera_v1.1_2023_rq1_pilot")
pilot["valid"] = ~np.isnan(pilot_emb).any(axis=1)
print(f"{len(clean)} presences; {len(expert)} expert polygons, {expert.in_district.sum()} in the district; "
      f"{len(pilot)} stored pixels, {pilot.valid.sum()} with a valid embedding")

## 1. RQ1: expert polygons with a stored embedding

A polygon enters the effort-reduction curve only if at least one
stored pixel centre with a valid embedding lies inside it.

In [ ]:
pts = gpd.GeoDataFrame(pilot, geometry=gpd.points_from_xy(pilot.x, pilot.y), crs=32629)
px = gpd.sjoin(pts[pts.valid], expert[["geometry", "in_district"]], predicate="within", how="inner")
px = px[~px.index.duplicated()].rename(columns={"index_right": "polygon"})
print(f"{px[px.in_district].polygon.nunique()} of the {expert.in_district.sum()} district polygons hold a stored pixel "
      f"({len(px[px.in_district])} pixels)")

## 2. RQ2: what the grade and accuracy filters remove

Inside the 2021–2025 window, *A. dealbata* only. The pixels added by
the removed observations are those the unfiltered condition needs.

In [ ]:
raw = pd.read_csv(ROOT / "data/raw/citizen_science/inaturalist_acacia_coimbra_pilot.csv")
raw["year"] = pd.to_datetime(raw.observed_on, errors="coerce").dt.year
window = raw[raw.taxon_name.str.startswith("Acacia dealbata", na=False) & raw.year.between(2021, 2025)]
obscured = window.coordinates_obscured.astype(bool)
located = window[~obscured]
bad_grade = located.quality_grade != "research"
graded = located[~bad_grade]
bad_accuracy = ~(graded.positional_accuracy.isna() | (graded.positional_accuracy <= 30))
print(f"{len(window)} observations in the window; {obscured.sum()} obscured; "
      f"{bad_grade.sum()} removed by grade {located[bad_grade].quality_grade.value_counts().to_dict()}; "
      f"{bad_accuracy.sum()} removed by accuracy; {len(graded[~bad_accuracy])} left")

removed = pd.concat([located[bad_grade], graded[bad_accuracy]])
removed["pixel"] = pixel_id(gpd.GeoDataFrame(removed, geometry=gpd.points_from_xy(removed.longitude, removed.latitude), crs=4326))
added = set(removed.pixel) - set(clean.pixel)
print(f"the two filters remove {len(removed)} observations, which add {len(added)} pixels; "
      f"{len(added & set(pilot[pilot.valid].pixel))} of them have a valid stored embedding")

## 3. RQ2: background points inside the mask window

Region growing may label up to 5 pixels from the observed pixel along
each axis (an 11 × 11 window, corners 71 m away), while the background
pool excludes 50 m around presences. This counts the background points
whose pixel lies inside the window of one of the presences.

In [ ]:
presence_cells = clean.pixel.str.split("_", expand=True).astype(int).to_numpy()
for kind in ("uniform", "forest_only"):
    bg = load_background(ROOT, kind)
    cells = np.column_stack([np.floor(bg["x"] / 10), np.floor(bg["y"] / 10)]).astype(int)
    steps, _ = cKDTree(presence_cells).query(cells, p=np.inf)  # largest offset in pixels, along either axis
    print(f"{kind}: {(steps <= 5).sum()} of {len(cells)} background points inside a window")

## 4. RQ3: near and far expert polygons

Distance from each expert polygon to the nearest of the presences;
near is up to 1 km. The test set uses the district polygons with a
stored embedding, without the pixels that are also the pixel of a
citizen-science presence. The counts per seed (distance to the
training presences only) are not made here.

In [ ]:
cs = gpd.GeoDataFrame(clean, geometry=gpd.points_from_xy(clean.longitude, clean.latitude), crs=4326).to_crs(32629)
dist = gpd.sjoin_nearest(expert[["geometry"]], cs[["geometry"]], distance_col="d")
dist = dist[~dist.index.duplicated()]["d"]
near = dist <= 1000
far_d = dist[~near & expert.in_district]
print(f"near: {near.sum()} polygons ({(near & expert.in_district).sum()} in the district); "
      f"far, in the district: {len(far_d)}, between {far_d.min() / 1000:.1f} and {far_d.max() / 1000:.1f} km")

test = px[px.in_district].copy()
test["near"] = near.reindex(test.polygon).values
test["shared"] = test.pixel.isin(set(clean.pixel))
print(f"expert pixels that are also a presence pixel: {test.shared.sum()} (near: {test[test.shared].near.sum()})")
(test[~test.shared].assign(group=lambda d: np.where(d.near, "near", "far"))
     .groupby("group").agg(polygons=("polygon", "nunique"), pixels=("pixel", "size")))